# Project - First Draft
this file is for experimenting, writing first bits of code and testing

In [ ]:

# import important packages
import requests
import pandas as pd
import geopandas as gpd
import time
import matplotlib.pyplot as plt
import folium
import contextily
import cmcrameri


from cartopy import crs as ccrs
from geodatasets import get_path

c:\Users\isabe\miniconda3\envs\sds-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# test my MAP_KEY

map_key = "ea49e5fe6beaf3a00954c71727386596"

import pandas as pd
import requests
key_url = f"https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?map_key={map_key}"
try:
  response = requests.get(key_url)

  if response.status_code == 200:
    data = response.json()
    df = pd.Series(data)
    display(df)
  else:
    print(f"Error in the query: HTTP {response.status_code}")
    print(response.text)

  
except Exception as e:
  # possible error, wrong MAP_KEY value, check for extra quotes, missing letters
  print (f"There is an issue with the query: {e}\n try in your browser: {key_url}")
  


transaction_limit             5000
current_transactions             0
transaction_interval    10 minutes
dtype: object

### Query API and Import Data

In [14]:
import requests
import time
import datetime

# Define Dates: start and enddate format (YYYY-MM-DD), oldest date 2017?
startdate = "2023-10-22"
enddate = "2023-10-26"


type(startdate)
type(enddate)

start = pd.to_datetime(startdate)
end = pd.to_datetime(enddate)

difference = end-start
print(difference)

days = (end-start).days
print (days)


4 days 00:00:00
4


#### first try to load data

In [ ]:

import requests
import time
import datetime
import geopandas as gpd
import pandas as pd

map_key = "ea49e5fe6beaf3a00954c71727386596"

# Define Dates: start and enddate format (YYYY-MM-DD), oldest date 2017?
startdate = "2023-10-22"
enddate = "2023-10-26"

# calculate days out of dates --> FIRMS uses a number of days in the API-call
start = pd.to_datetime(startdate)
end = pd.to_datetime(enddate)

days = (end-start).days

# transform dates to FIRMS accepted format (YYYY-MM-DD)

# for loop später

#area_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_NOAA20_NRT/world/1'

#date.today() fetches the current date -> is a date-object
# srftime converts dateobject to strings as FIRMS needs it in a string format (YYYY-MM-DD)

# source for this???


# define coordinates for South America

region_coords = "-55.0,-81.5,12.5,-35.0"

# define sensor and products
sensor = "VIIRS_NOAA20_SP"
product = "fire"

# define API endpoint url

api_url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{map_key}/{sensor}/{region_coords}/{days}"


# define query parameters
parameters = {
    "key": map_key,
    "start_date": startdate,
    "end_date": enddate,
    "area": region_coords,
    "sensor": sensor,
    "product": product,
    "format": "csv"    
}


# fetch the data

response = requests.get(api_url, params = parameters)

# check the status and extract the data
if response.status_code == 200: 
    print("Request successful\n")

    # read url and create dataframe
    df_fire = gpd.read_file(api_url)
    print(f"\n Data download successful! {len(df_fire)} fires found")  ## NA handling??
    display(df_fire.head(6))

    # option: include a saving option if wanted
    #df_fire.to_csv("firms_data_startdate_enddate.csv", index=False) wieso index = false??

elif response.status_code == 404:
    print(f"Error 404: Page not found")

elif response.status_code == 401:
    print(f"Error 401: Invalid API-Key, please check map-key")

elif response.status_code == 400:
    print(f"Error 400: Wrong parameters, please check input parameters.")


else: 
    print(f"Request failed: Status {response.status_code}")
    print(response.text)


Error 400: Wrong parameters, please check input parameters.


#### second try to load data; works but only fetches the last 8 days

current FIRMS-URL fetches only the data for the last 8 days. It does not consider start and enddate as I calculated it. Have to create a workaround so api accepts it

In [ ]:

import requests
import time
import datetime import timedelta
import geopandas as gpd
import pandas as pd

map_key = "ea49e5fe6beaf3a00954c71727386596"

# Define Dates: start and enddate format (YYYY-MM-DD), oldest date 2017?
startdate = "2024-09-22"
enddate = "2024-09-30"

# calculate days out of dates --> FIRMS uses a number of days in the API-call
start = pd.to_datetime(startdate)
end = pd.to_datetime(enddate)

days = (end-start).days

# transform dates to FIRMS accepted format (YYYY-MM-DD)

# for loop später

#area_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_NOAA20_NRT/world/1'

#date.today() fetches the current date -> is a date-object
# srftime converts dateobject to strings as FIRMS needs it in a string format (YYYY-MM-DD)

# source for this???


# define coordinates for South America

region_coords = "-55.0,-81.5,12.5,-35.0"

# define sensor and products
sensor = "VIIRS_NOAA20_SP"
product = "fire"

# define API endpoint url

api_url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{map_key}/{sensor}/{region_coords}/{days}"


# take parameters out, because firms reads everything out of the url


# fetch the data

response = requests.get(api_url)

# check the status and extract the data
if response.status_code == 200: 
    print("Request successful\n")

    # read url and create dataframe
    df_fire = pd.read_csv(api_url)
    print(f"\n Data download successful! {len(df_fire)} fires found")  ## NA handling??
    display(df_fire.head(6))

    # option: include a saving option if wanted
    #df_fire.to_csv("firms_data_startdate_enddate.csv", index=False) wieso index = false??

elif response.status_code == 404:
    print(f"Error 404: Page not found")

elif response.status_code == 401:
    print(f"Error 401: Invalid API-Key, please check map-key")

elif response.status_code == 400:
    print(f"Error 400: Wrong parameters, please check input parameters.")


else: 
    print(f"Request failed: Status {response.status_code}")
    print(response.text)


Request successful


 Data download successful! 0 fires found


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,type


In [ ]:
#lumo

import pandas as pd
import requests
from datetime import date

# --- KONFIGURATION ---
API_KEY = "ea49e5fe6beaf3a00954c71727386596"  # Ersetze dies mit deinem gültigen Key
BASE_URL = "https://firms.modaps.eosdis.nasa.gov/api/area/"

# 1. Datum festlegen (Heute)
# FIRMS benötigt das Format YYYY-MM-DD
today = date.today().strftime("%Y-%m-%d")

# 2. Region definieren (Beispiel: Ein Rechteck über Deutschland)
# Format: lat_min,lon_min,lat_max,lon_max
# Oder als GeoJSON Polygon-String
region_coords = "47.0,5.0,55.0,15.0"  # Beispiel: Südwest bis Nordost Deutschland

# 3. Sensor und Produkt wählen
# Sensoren: 'VIIRS', 'MODIS', 'SNPP'
# Produkte: 'fire' (Feuer), 'thermal_anomalies' (Thermische Anomalien)
sensor = "VIIRS"
product = "fire"

# --- API AUFRUF ---
params = {
    "key": API_KEY,
    "date": today,
    "area": region_coords,
    "sensor": sensor,
    "product": product,
    "format": "csv"  # FIRMS liefert oft CSV zurück, das ist einfacher zu parsen
}

print(f"Abfrage für Datum: {today}, Region: {region_coords}")

try:
    response = requests.get(BASE_URL, params=params)
    
    # Statuscode prüfen
    if response.status_code == 200:
        # Die API gibt oft CSV zurück, nicht JSON
        # Wir nutzen pandas, um das CSV direkt zu lesen
        df = pd.read_csv(pd.io.common.StringIO(response.text))
        
        print(f"\nErfolgreich! {len(df)} Feuerstellen gefunden.")
        display(df.head()) # Zeige die ersten 5 Zeilen
        
        # Optional: Speichern
        # df.to_csv("firms_data_today.csv", index=False)
        
    elif response.status_code == 401:
        print("Fehler 401: Ungültiger API-Key. Bitte überprüfe deinen Key.")
    elif response.status_code == 400:
        print("Fehler 400: Falsche Parameter. Prüfe Datum und Koordinaten.")
        print(f"URL: {response.url}")
        print(f"Inhalt: {response.text[:200]}")
    else:
        print(f"Fehler: Status {response.status_code}")
        print(response.text)

except Exception as e:
    print(f"Ein unerwarteter Fehler trat auf: {e}")

Abfrage für Datum: 2026-05-07, Region: 47.0,5.0,55.0,15.0
Ein unerwarteter Fehler trat auf: Error tokenizing data. C error: Expected 1 fields in line 7, saw 13

